In [8]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Point


In [9]:
ACS_CSV = "external/acs_la_tract_2022.csv"
TRACTS_GEO = "external/la_tracts.geojson"
CRIME_CSV = "Crime_Data_from_2020_to_Present copy.csv"

In [10]:
acs = pd.read_csv(ACS_CSV, dtype={"GEOID": str})
tracts = gpd.read_file(TRACTS_GEO)
crime = pd.read_csv(CRIME_CSV)
acs.head(2), tracts.head(2), crime.head(2)

(        geoid  total_pop  median_income  state  county   tract  \
 0  6037101110       4014        68972.0      6      37  101110   
 1  6037101122       4164       118859.0      6      37  101122   
 
                              name  b01003_001_moe  b19013_001_moe  
 0  Los Angeles County, California           473.0         17023.0  
 1  Los Angeles County, California           822.0         31445.0  ,
          GEOID  B01003_001E state county   tract  \
 0  06037670413         4852    06    037  670413   
 1  06037650101         5532    06    037  650101   
 
                              NAME  B01003_001_moe  \
 0  Los Angeles County, California           540.0   
 1  Los Angeles County, California           906.0   
 
                                             geometry  
 0  POLYGON ((-118.4037 33.78136, -118.39236 33.78...  
 1  POLYGON ((-118.32645 33.8728, -118.31779 33.87...  ,
        DR_NO               Date Rptd                DATE OCC  TIME OCC  AREA  \
 0  211507896 

In [11]:
def to_snake(s): 
    return s.strip().lower().replace(" ", "_").replace("-", "_")

def standardize(df): 
    df = df.copy(); df.columns = [to_snake(c) for c in df.columns]; return df

In [12]:
acs = standardize(acs).rename(columns={"b01003_001e":"total_pop","b19013_001e":"median_income"})
tracts = standardize(tracts)
crime = standardize(crime)

In [13]:
# ensure GEOID present
if "geoid" not in tracts.columns:
    for cand in ["geoid10","GEOID","tractce","TRACTCE"]:
        if cand.lower() in tracts.columns:
            tracts = tracts.rename(columns={cand.lower():"geoid"}); break


In [14]:
# minimal crime cleaning (fast)
crime = crime.rename(columns={"area_name":"area_name"})  # no-op; keep if exists
needed = ["date_occ","crm_cd_desc","vict_age","lat","lon"]
for c in needed:
    if c not in crime.columns:
        pass  # continue; this notebook focuses on spatial join

In [15]:
# keep only valid coords
crime = crime.dropna(subset=["lat","lon"])
crime = crime[(crime["lat"].between(-90,90)) & (crime["lon"].between(-180,180))]

In [16]:
# completeness snapshot (for rubric)
display(acs[["geoid","total_pop","median_income"]].isnull().sum())


geoid             0
total_pop         0
median_income    49
dtype: int64

In [17]:
# Merge tract and acs
tracts_acs = tracts.merge(acs[["geoid","total_pop","median_income"]], on="geoid", how="left")

cols_to_drop = ["b01003_001e","b01003_001e_moe","state","county","tract","name"]
tracts_acs.drop(columns=[c for c in cols_to_drop if c in tracts_acs.columns], inplace=True, errors="ignore")
tracts_acs = tracts_acs[["geoid","total_pop","median_income","geometry"]]
tracts_acs.head(2)

ValueError: You are trying to merge on object and int64 columns for key 'geoid'. If you wish to proceed you should use pd.concat